<style>
.jp-Notebook { max-width: 1180px; margin: auto; }
.jp-RenderedHTMLCommon h1 { color: #17365d; }
.jp-RenderedHTMLCommon h2 { color: #245b86; margin-top: 1.4em; }
.jp-RenderedHTMLCommon table { font-size: 0.95em; }
</style>

# Deeply Temporalised Life Cycle Assessment with Climate Emulator Coupling

**Add timing to LCA -- from a simple car model to direct air capture and storage using Premise data package**

Romain Sacchi · Laboratory for Energy Systems Analysis · PSI  
BrightCon 2026

> Our research question: When do life-cycle exchanges occur, and which background system supplies them?

By the end, we should be able to:

1. explain how TRAILS places exchanges in time,
2. why this can change an LCA result,
3. and timed inventory can be used in three impact assessments approaches.

## What TRAILS adds

TRAILS stands for **Temporal Routing And Aggregation of Impacts across Life-cycle Systems**.  
It combines the timing of life-cycle exchanges with a background system that changes over time.  
You supply the exchange timing and the background scenarios; TRAILS uses them to calculate the inventory.

There are three main steps:

```text
Functional unit and reference year
      ↓
temporal_routing() — assign exchanges to calendar years
      ↓
lci() — solve the supply-chain demands for each year
      ↓
Inventory by activity, elementary flow and year
      ├── lcia(), e.g., with GWP100
      ├── FaIR: radiative forcing and temperature change
      └── lcia() with EDGES (exchange-based characterisation): regional water scarcity
```

We keep three ideas separate: **when an exchange occurs**, **which year's background supplies it**,
and **how its impact is assessed**.  

We start with a small car model so that we can inspect the inputs before looking at the results.

### Setup

In [ ]:
import json
import warnings
from importlib.resources import files
from pathlib import Path

import pandas as pd
from datapackage import Package

import edges
import trails as trails_package
from trails import Trails, plot_temp, plot_temporal_scores, search_activity
from trails.fair_rf import run_fair_delta_rf

warnings.filterwarnings("ignore", category=FutureWarning)
print(f"TRAILS {trails_package.__version__}")

In [ ]:
# define some paths
ROOT = Path.cwd().resolve()
DATA = ROOT / "data"
SMALL_PACKAGE = DATA / "example_data_package" / "datapackage.json"
PREMISE_PACKAGE = DATA / "trails_remind_SSP2-PkBudg1000.zip"
DACCS_INVENTORY = DATA / "lci-case-study-daccs_storage_risk.xlsx"

In [ ]:
def prospective_aware_method(ssp: str = "SSP126") -> dict:
    path = files("edges").joinpath(
        "data/AWARE 2.0 prospective_Country_all_yearly.json"
    )
    method = json.loads(path.read_text(encoding="utf-8"))
    if ssp not in method["parameters"]:
        raise ValueError(f"Unknown AWARE scenario {ssp!r}")
    method["parameters"] = {ssp: method["parameters"][ssp]}
    method["name"] = f"AWARE 2.0 prospective | Country | all | yearly | {ssp}"
    return method

# 1. A simple car example

The data package contains **21 activities**, with background data for **2005, 2020, 2050 and 2100**.
TRAILS interpolates between these years to obtain annual background matrices.

**Functional unit:** 200,000 vehicle-kilometres, with vehicle use starting in 2050.
This is a small teaching model, not a representative assessment of a real car.

We load the package and choose GWP100. The full list of available methods is for reference.

In [ ]:
car = Trails(
    Package(str(SMALL_PACKAGE)),
    ei_version="3.12",
)

In [ ]:
CAR_GWP = "IPCC 2021 (incl. biogenic CO2) - climate change: total (incl. biogenic CO2) - global warming potential (GWP100)"

In [ ]:
from trails import get_lcia_method_names
get_lcia_method_names()

### Inspect the background matrices

`A` contains technosphere exchanges; `B` contains elementary flows.
Each has an extra axis for the background scenario year. The other axes describe
activities and products (`A`), or activities and elementary flows (`B`).

The number of years shown below includes interpolation; it is not the number of original input years.

In [ ]:
print("A[year, activity, activity]:", car.A.shape)
print("B[year, activity, flow]:    ", car.B.shape)

### Find the car activity

Search by name and check which activity represents the functional unit.

In [ ]:
search_activity(car, name="passenger car")

### Read the car's exchange timing

In this example data package, activity `13` is the internal-combustion-engine car.
The table below shows its exchanges and their timing. Offsets are measured **in years** from the activity's reference year.

Focus on three rows: fuel use is spread over offsets **0–16**; vehicle production has no explicit delay;
end-of-life is split between offsets **17 and 18**, with weights **0.3 and 0.7**.
For a 2050 start, those end-of-life exchanges occur in 2067 and 2068.

`port` spreads the reference-year exchange amount over time. `matrix` reads the exchange coefficient
from the background matrix for each pulse year. These choices can affect both the timing and the total inventory.

In [ ]:
car_idx = 13
car.print_exchange_table(year=2050, act_idx=car_idx)

## Place the life cycle in time

`temporal_routing()` follows the exchanges and assigns demands and direct emissions to calendar years.  
The next cell uses GWP100 to decide how far to follow each branch in detail.

The cutoff controls this detailed traversal. A branch that stops is still included in the subsequent matrix solve;
its remaining supply chain is not discarded.  
The graph cell then writes an HTML view that you can open to inspect the timing.

In [ ]:
car.temporal_routing(
    start_year=2050,
    start_act_idx=car_idx,
    amount=200_000,
    adaptive_methods=[CAR_GWP],
    adaptive_relative_score_cutoff=0.00001
)

In [ ]:
from trails.plotting import plot_temporal_graph

plot_temporal_graph(
    car,
    filename='trails_graph.html',
)

### Calculate the inventory

`lci()` solves the demands for each active year using the corresponding background matrix.
It retains the resulting inventory by year. No characterization factors are applied yet.

In [ ]:
car.lci()

### Apply GWP100

`lcia()` applies GWP100 to the inventory. We can apply another method later
without calculating the inventory again.

In [ ]:
car_gwp_scores = car.lcia(methods=[CAR_GWP])

### Calculate a static reference

For comparison, we calculate the same functional unit using the 2050 background for the whole life cycle.
This is a prospective static reference: “static” does not mean “present-day”.

In [ ]:
car.static_lca(
    year=2050,
    act_idx=car_idx,
    methods=[CAR_GWP],
    amount=200_000,
)

### Compare the temporal and static results

The plot shows GWP100 contributions by inventory year and their cumulative total.
The red dotted line is the static 2050 result.

In [ ]:
plot_temporal_scores(
    trails=car,
    stacked=False,
    legend_top_n=7,
    show_flow_contributions=False,
    title="Passenger car: temporal versus static GWP",
    method_label="kg CO₂-eq",
    year_range=(2020, 2080),
    year_tick=10,
    reference_year=2050,
    show_cumulative_axis=True,
    static_score=car.static_score,
    static_score_dash="dot",
    static_score_color="red",
    width=850,
    height=480,
)

# 2. Apply the same workflow to DACCS (data package not provided)

**DACCS** means direct air capture with carbon storage.
We now use a larger background package produced with `Premise`, based on the **REMIND SSP2-PkBudg1000** scenario,
and import an Excel inventory for solvent-based DACCS.  
`Premise` has introduce temproal disitrbution through the background database exchanges.

**Functional unit:** 20 Mt of CO₂ captured over a nominal 20-year operating period, starting in 2035.  
This is gross capture, not net climate benefit: life-cycle emissions also enter the assessment.  
The supplied inventory assigns operation to annual offsets **0–20**, or 2035–2055 inclusive.

The workflow is the same as for the car: load → route → calculate the inventory → assess impacts.

In [ ]:
GWP = "IPCC 2021 (incl. biogenic CO2) - climate change: total (incl. biogenic CO2) - global warming potential (GWP100)"

START_YEAR = 2035
DACCS_AMOUNT_KG = 20_000_000_000.0

### Load the future background

TRAILS loads the scenario package and, by default, interpolates the background matrices to annual resolution.
The `Trails?` cell opens the API help in Jupyter; it is optional during the talk.

In [ ]:
Trails?

In [ ]:
%%time
PREMISE_PACKAGE="/Users/romain/GitHub/trails/dev/publication/trails_remind_SSP2-PkBudg1000.zip"
daccs = Trails(
    Package(str(PREMISE_PACKAGE))
)

In [ ]:
# let's check the shape of A and B
print("Shape of A", daccs.A.shape)
print("Shape of B", daccs.B.shape)

In [ ]:
# let's check the number of temporal distribution
print("Temporalised biosphere exchanges", len(daccs.temporal_biosphere_exchanges))
print("Temporalised technosphere exchanges", len(daccs.temporal_technosphere_exchanges))

### Add the DACCS inventory

The Excel file supplies foreground exchanges and their timing.
TRAILS links these exchanges to activities and elementary flows in the background package.

In [ ]:
daccs.import_excel_inventory(str(DACCS_INVENTORY))

### Find the DACCS activity

Search by dataset name and location, then check the returned activity before using its index.
The search below filters by text; it does not enforce a unique exact match.

In [ ]:
search_activity(
    daccs,
    name="carbon dioxide, captured, with a solvent-based direct air capture system, 1MtCO2",
    location="Europe"
)

### Check the selected activity

The saved example uses activity index `42300`. Check it against the search result if the package or import changes.  
In the exchange table, construction occurs before the 2035 start, operation is spread over offsets 0–20,
and end-of-life occurs at offset 21. Check the production amount as well as the exchange amounts.

In [ ]:
daccs_idx=42300
daccs.print_exchange_table(
    year=2035,
    act_idx=daccs_idx
)

### Place the DACCS exchanges in time

We use the same routing step as for the car. GWP100 guides the detailed traversal.
`attribute_to_roots=True` keeps track of contributions associated with the starting activity's direct exchanges,
such as its heat and electricity supply. The later plots use this grouping.

The following graph cell exports an HTML view of the routed system.

In [ ]:
daccs.temporal_routing(
    start_year=START_YEAR,
    start_act_idx=daccs_idx,
    amount=DACCS_AMOUNT_KG,
    attribute_to_roots=True,
    adaptive_methods=[GWP],
)

In [ ]:
from trails.plotting import plot_temporal_graph

plot_temporal_graph(
    daccs,
    filename='trails_graph_daccs.html',
)

### Calculate and keep the inventory

We solve the annual demands once and retain the inventory for GWP100, FaIR and AWARE.
This larger calculation can take time; use the saved results when presenting if necessary.

In [ ]:
daccs.lci(
    solver_mode="iterative",
    iterative_rtol=1e-3,
)

### Apply GWP100 and calculate a static reference

We first apply fixed GWP100 factors to the annual inventory.
We then calculate a static reference using the 2035 background for the whole life cycle.

In [ ]:
gwp_scores = daccs.lcia(methods=[GWP])

In [ ]:
daccs.static_lca(
    year=2035,
    act_idx=daccs_idx,
    methods=[GWP],
    amount=DACCS_AMOUNT_KG,
)

### Read the DACCS climate result

The plot groups annual GWP100 contributions by their first-level supply-chain origin.
Negative contributions reduce the total; positive contributions increase it.  
Compare the cumulative result with the static reference.

The displayed range ends in 2070. It is a view of the results, not a definition of the full inventory horizon.

In [ ]:
static_gwp_score = float(daccs.static_score[0])
static_gwp_score

In [ ]:
plot_temporal_scores(
    trails=daccs,
    stacked=True,
    legend_top_n=7,
    show_flow_contributions=False,
    title="DACCS: life-cycle GWP through time",
    method_label="kg CO₂-eq",
    year_range=(2020, 2070),
    year_tick=5,
    reference_year=START_YEAR,
    show_cumulative_axis=True,
    width=850,
    height=480,
    static_score=daccs.static_score
)

# 3. But GWP isn't a very fitting indicator here...

The sliding time window of the GWP indicator is not ideal for decision-making.

## A. Calculate the climate response with FaIR

GWP100 gives a fixed 100-year equivalence for each gas.  
It does not show the resulting temperature trajectory.

Instead, we use the climate model `FaIR`.  
Here, it compares a **background emissions pathway** with that pathway **plus the TRAILS inventory**.  
The difference is the climate response attributed to the inventory.  

We provide the climate model with the background emissions as calculated in the REMIND pathway used by `Premise` to ensure consistency.

In [ ]:
run_fair_delta_rf(
    daccs,
    scenario="REMIND|SSP2-PkBudg1000"
)

### Read the radiative-forcing and temperature plots

The first plot shows the change in **radiative forcing** (W/m²); the second shows the change in **temperature** (°C).  
Both show median responses through 2200, grouped by first-level contribution.  
Negative values indicate lower forcing or temperature than in the background run.  

In [ ]:
from trails.plotting import plot_rf

plot_rf(
    daccs,
    #by="root activity",
    by="flow",
    flow_groupby_name=True,
    title="DACCS: RF response of timed emissions and removals",
    method_label="W/m²",
    year_range=(2020, 2200),
    year_tick=20,
    width=850,
    height=480,
)

In [ ]:
plot_temp(
    daccs,
    by="root activity",
    title="DACCS: temperature response of timed emissions and removals",
    method_label="°C",
    year_range=(2020, 2200),
    year_tick=20,
    width=850,
    height=480,
)

### Optional extension: express the response as a CO₂ pulse equivalent

How large would a single CO₂ pulse need to be to produce the same **integrated radiative forcing** as this inventory?  
We compare the area under the inventory's forcing curve with the area under the curve for a known CO₂ pulse:

$$M_{\mathrm{CO_2,eq}} = M_{\mathrm{ref}}\,\frac{\int_{t_0}^{t_1} \Delta RF_{\mathrm{LCA}}(t)\,dt}{\int_{t_0}^{t_1} \Delta RF_{\mathrm{CO_2\ pulse}}(t)\,dt}.$$

The code below places the reference pulse in **2026** and integrates both responses over **2000–2100**.  
These are metric choices; the DACCS activity still starts in **2035**.  
The **1 Mt CO₂** reference pulse sets the scale of the comparison.

TRAILS calculates a ratio for each FaIR configuration before summarizing the ensemble.  
A negative equivalent means a net reduction in integrated forcing over this window.  
This is a window-specific RF metric, not GWP100.

In [ ]:
from trails.fair_rf import run_fair_co2_pulse_equivalents

co2_pulse_result = run_fair_co2_pulse_equivalents(
    daccs,
    scenario="REMIND|SSP2-PkBudg1000",
    reference_pulse_year=2026,
    window_start=2000,
    window_end=2100,
    reference_pulse_mass_kg=1.0e9,  # 1 Mt CO2
)

rf_equivalent = co2_pulse_result["co2_pulse_equivalent"]["integrated_rf"]
rf_equivalence_summary = pd.DataFrame(
    {
        "median": [rf_equivalent["median"] / 1.0e9],
        "2.5%": [rf_equivalent["p025"] / 1.0e9],
        "97.5%": [rf_equivalent["p975"] / 1.0e9],
    },
    index=["Integrated RF CO2 pulse-equivalent [Mt CO2]"],
)
rf_equivalence_summary

#### Estimate gross capture per tonne of RF-equivalent removal

For a negative RF equivalent, we can estimate how much gross capture corresponds to the benefit
of a **one-tonne CO₂ removal pulse in 2026**, assessed over **2000–2100**:

$$M_{\mathrm{DACCS,required}} = 1\ \mathrm{t\ CO_2}\times\frac{M_{\mathrm{DACCS,modelled}}}{\left|M_{\mathrm{CO_2,eq}}\right|}.$$

This scaling assumes that the inventory and its climate benefit scale approximately linearly with DACCS deployment.  

In [ ]:
target_co2_equivalent_kg = 1_000.0  # 1 tonne CO2
modelled_daccs_kg = DACCS_AMOUNT_KG

required_daccs_summary = pd.DataFrame(
    {
        "median": [
            target_co2_equivalent_kg
            * modelled_daccs_kg
            / abs(rf_equivalent["median"])
            / 1_000
        ],
        "2.5%": [
            target_co2_equivalent_kg
            * modelled_daccs_kg
            / abs(rf_equivalent["p025"])
            / 1_000
        ],
        "97.5%": [
            target_co2_equivalent_kg
            * modelled_daccs_kg
            / abs(rf_equivalent["p975"])
            / 1_000
        ],
    },
    index=["Gross DACCS required per 1 t RF-equivalent removal [t CO2]"],
)
required_daccs_summary

Read the freshly calculated table as **tonnes of gross CO₂ capture per tonne of RF-equivalent removal**.  
The result depends on the reference pulse year and the integration window.  

It is not a physical storage efficiency or a GWP100 result. It includes the effects of life-cycle emissions,
removals, non-CO₂ gases and their timing.   
The percentile range describes variation across the FaIR configurations.


## B. Assess water scarcity with AWARE and EDGES

**AWARE** assesses the potential to deprive other users of water.  
**EDGES** applies characterization factors to exchanges, using information about the flow and the associated activity.  
This lets water-use factors vary with the activity's location and the inventory year.  

Here we select country-level prospective AWARE 2.0 factors under **SSP2-RCP 2.6**, to align with the foreground inventory scenario (**REMIND SSP2-PkBudg1000**).

In [ ]:
aware_method = prospective_aware_method("SSP126")

### Compare water-scarcity factors

The table compares non-irrigation water factors for **Switzerland, Spain and Denmark** in **2024 and 2044**, acoording to AWARE 2.0: SSP1-RCP 2.6.
For the same volume of water, a larger factor gives a larger water-scarcity score.

In [ ]:
parameters = aware_method["parameters"]["SSP126"]

pd.DataFrame(
    {
        "Switzerland": [parameters[f"cf_non_irri_ch"][str(year)] for year in (2024, 2044)],
        "Spain": [parameters[f"cf_non_irri_es"][str(year)] for year in (2024, 2044)],
        "Denmark": [parameters[f"cf_non_irri_dk"][str(year)] for year in (2024, 2044)],
    },
    index=pd.Index([2024, 2044], name="year"),
).rename_axis(columns="m³ deprived / m³ water")

### Apply AWARE to the inventory

We call `lcia()` again, this time using EDGES and the selected AWARE factors.
The annual inventory is reused; we do not repeat the supply-chain solves.

In [ ]:
aware_scores = daccs.lcia(
    methods=[aware_method],
    method_backend="edges",
    reuse_mappings=True,
)

### Read the water-scarcity result

The plot groups water-scarcity contributions by year and first-level supply-chain origin.
Both water use and the country- and year-specific factors can affect the score.  
The score does not reflect a physical volume of freshwater consumed.  
Rather, itis a characterized scarcity impact (i.e., m3-deprived-eq.).

In [ ]:
plot_temporal_scores(
    trails=daccs,
    stacked=True,
    legend_top_n=7,
    show_flow_contributions=False,
    title="DACCS: time- and location-sensitive impact assessment using AWARE 2.0",
    method_label="m³ deprived water-eq.",
    year_range=(2030, 2060),
    year_tick=5,
    reference_year=START_YEAR,
    show_cumulative_axis=True,
    width=850,
    height=480,
)

## What to take away

TRAILS combines **exchange timing** with **background systems that change over time**.
Its central workflow is `temporal_routing()` → `lci()` → `lcia()`.  
The same annual inventory can also support a `FaIR` climate-response calculation or an integration with `Edges`.


## References for the tools and methods used

- **TRAILS — temporal life-cycle modelling.** Sacchi, R., et al. (2026). [A graph-matrix hybrid approach for deep temporalisation in time-explicit LCA](https://doi.org/10.21203/rs.3.rs-10139523/v1). *Research Square*, preprint, version 1.

- **EDGES — exchange-level impact assessment.** Sacchi, R., et al. (2025). [Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment](https://doi.org/10.1007/s11367-025-02551-7). *The International Journal of Life Cycle Assessment*, **30**, 3087–3101.

- **FaIR — radiative forcing and temperature response.** Leach, N. J., et al. (2021). [FaIRv2.0.0: a generalized impulse response model for climate uncertainty and future scenario exploration](https://doi.org/10.5194/gmd-14-3007-2021). *Geoscientific Model Development*, **14**, 3007–3036.

- **AWARE 2.0 — prospective water-scarcity factors.** Seitfudem, G., Berger, M., & Boulay, A.-M. (2026). [Harnessing model ensembles to assess uncertainty and provide prospective characterization factors for AWARE2.0](https://doi.org/10.1007/s11367-026-02648-7). *The International Journal of Life Cycle Assessment*, **31**, article 98.
